# Evaluación final (Fase 5): modelo D sobre test set

Carga el modelo final entrenado (best.pt subido como Kaggle Dataset) y lo evalúa sobre el split test del dataset epp-yolo-processed. Reporta mAP global y por clase + matriz de confusión + curvas P/R.

> **Notebook delgado**: solo orquesta. La lógica vive en `src/` del repositorio.

**Requisitos:**
- Adjuntar el dataset `epp-yolo-processed` como Input.
- Adjuntar un dataset que contenga el `best.pt` del modelo a evaluar.

In [ ]:
# Setup en Kaggle: clona el repo del proyecto e instala dependencias.
# El código pesado vive en `src/`; este notebook solo orquesta.
import os, subprocess, sys

REPO_URL = "https://github.com/PercyTechX/epp-yolo-compliance.git"
REPO_DIR = "/kaggle/working/repo"

if not os.path.isdir(REPO_DIR):
    subprocess.check_call(["git", "clone", "--depth", "1", REPO_URL, REPO_DIR])
os.chdir(REPO_DIR)
subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"])

In [ ]:
# Evaluación final del modelo elegido sobre el split test (no visto).
# Auto-detecta tanto la ruta del Kaggle Dataset como el best.pt en /kaggle/input/.
import os, yaml, json
from pathlib import Path
from ultralytics import YOLO


def find_dataset_root() -> str:
    """Localiza la raíz del Kaggle Dataset processed (images/+labels/)."""
    for root, dirs, _ in os.walk("/kaggle/input"):
        if "images" in dirs and "labels" in dirs:
            if os.path.isdir(f"{root}/images/test"):
                return root
    raise RuntimeError("Dataset processed no encontrado bajo /kaggle/input/")


def find_weights() -> str:
    """Localiza un best.pt cualquiera bajo /kaggle/input/."""
    for root, _, files in os.walk("/kaggle/input"):
        if "best.pt" in files:
            return os.path.join(root, "best.pt")
    raise RuntimeError("best.pt no encontrado bajo /kaggle/input/")


DATA_ROOT = find_dataset_root()
WEIGHTS = find_weights()
print(f"Dataset montado en: {DATA_ROOT}")
print(f"Pesos:               {WEIGHTS}")

# Parchear data.yaml con la ruta real de Kaggle.
data_cfg = yaml.safe_load(Path("configs/data.yaml").read_text())
data_cfg["path"] = DATA_ROOT
patched = Path("/kaggle/working/data_patched.yaml")
patched.write_text(yaml.safe_dump(data_cfg, sort_keys=False))

model = YOLO(WEIGHTS)
metrics = model.val(
    data=str(patched),
    split="test",
    project="/kaggle/working/runs",
    name="test_eval_exp_D",
    save_json=True,
    plots=True,
)

# Resumen estructurado para descargar.
summary = {
    "mAP50": float(metrics.box.map50),
    "mAP50_95": float(metrics.box.map),
    "precision": float(metrics.box.mp),
    "recall": float(metrics.box.mr),
    "per_class": {
        str(cid): {
            "mAP50": float(metrics.box.maps[cid] if cid < len(metrics.box.maps) else 0.0),
        } for cid in range(len(data_cfg["names"]))
    },
}
Path("/kaggle/working/test_metrics.json").write_text(json.dumps(summary, indent=2))
print(json.dumps(summary, indent=2))